In [ ]:
!pip install -q transformers datasets accelerate huggingface_hub
import os
import torch
from datasets import load_dataset
from transformers import (
    VisionEncoderDecoderModel,
    TrOCRProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from huggingface_hub import login

# Use your actual Hugging Face write token
hf_token = "i-wont share token"
login(hf_token)

print("Authentication step initialized!")

model_id = "tjoab/latex_finetuned"
repo_id = "willy-wonka5745/Handwriting-to-LaTeX"

processor = TrOCRProcessor.from_pretrained(model_id)
model = VisionEncoderDecoderModel.from_pretrained(model_id)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id

#%%

print("Loading unsloth/LaTeX_OCR dataset...")
dataset = load_dataset("unsloth/LaTeX_OCR", split="train")

split_dataset = dataset.train_test_split(test_size=0.05)
train_data = split_dataset["train"]
eval_data = split_dataset["test"]


#%%

# Dynamic batch collator: processes only 4 images at a time on the fly


def collate_fn(batch):
    images = [item["image"].convert("RGB") for item in batch]
    texts = [item["text"] for item in batch]

    pixel_values = processor(images, return_tensors="pt").pixel_values

    labels = processor.tokenizer(
        texts,
        padding="longest",
        max_length=128,
        truncation=True,
        return_tensors="pt"
    ).input_ids

    # Replace padding token ids with -100 so cross-entropy loss ignores them
    labels[labels == processor.tokenizer.pad_token_id] = -100

    return {"pixel_values": pixel_values, "labels": labels}

#%%


training_args = Seq2SeqTrainingArguments(
    output_dir="./latex-ocr-checkpoints",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,         # Massive VRAM saver
    optim="adamw_torch_fused",           # Faster weight updates
    dataloader_num_workers=2,            # Eliminates GPU waiting time
    dataloader_pin_memory=True,          # Speeds up CPU-to-GPU data transfer
    predict_with_generate=False,
    fp16=True,
    learning_rate=4e-5,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    report_to="none",
    push_to_hub=True,
    hub_model_id=repo_id,
    hub_strategy="every_save",
)

#%%

trainer = Seq2SeqTrainer(
    model=model,
    processing_class=processor.tokenizer,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=collate_fn,
)

#%%

print("Beginning PyTorch training on full dataset...")
trainer.train(resume_from_checkpoint=True)

#%%


print(f"Pushing final weights to {repo_id}...")
trainer.push_to_hub()
processor.push_to_hub(repo_id)
print("Deployment complete!")